# EDA/SDA 1 (Lean): Field Comparison and Scope

PredictLeads × TripleTen externship · Jacob Colombo

## Purpose

Clean PredictLeads' full job-openings file (every listing first seen March 30 – September 30, 2026), compare fields, cut to the scope (**the roles TripleTen trains for, plus adjacent tech roles**) and explore it.

- **Q1 (section 3):** how do computer and mathematical roles compare with every other field?
- **Q2 (section 5):** which factors should drive prioritization for a career changer?
- **Q3 (section 6):** what time information can prioritization rely on?

Fix or flag, never drop, except data errors; aggregates only, never rows. **"Still open"** means last seen within 2 days of September 30.

**Hands off (section 7):** the scope table (every field) and its descriptions, the occupation summary, and company activity.

| Notebook | Question | Product |
|:--|:--|:--|
| **EDA_SDA_1** | **What's in the data, and what's the scope?** | **The scope table** |
| EDA_SDA_2 | Which scope listings do we keep, and how do they behave? | Cleaned and flagged scope table (ghosts, hiding rules) |
| EDA_SDA_3 | What do the descriptions say? | Text features and O*NET skills |
| EDA_SDA_4 | Which features hold up? | `modeling_ready.parquet` (the features that survived) |
| TimeSeries | When do listings appear and close? | Timing patterns by month |
| Profiles | Who really changed into or within tech or math, and how? (frozen) | The 2,813 career changers |
| ML_Fit | Which roles does a career changer fit? | The trained layer (role fit), saved for the app |
| ML_Rank | Which listings should they apply to first? | Listing scores and the queue's weights, saved for the app |
| Evaluate | How well do the app's rankers do on real career moves? | The comparison: `rrf`, `mlrank`, combined |

Same table as [`../PROJECT.md`](../PROJECT.md) ("The notebooks"), kept in step with it.

### About this Lean draft

A plain-pandas rewrite of this notebook: every cell uses only pandas/numpy/matplotlib and the project's real
data-loading code (`src/pl_data.py`, `src/rows_table.py`, `src/stream_full.py`) - no shared formatting or
summary-table helpers (`src/eda_helpers.py`). The goal is code that can be explained line by line, the way a
TripleTen project can, even where that costs more lines than the original. This will replace `EDA_SDA_1`
once finished - the plan is a Lean rewrite of every notebook in the chain, not a one-off. The standing rules
behind every call made here (what gets inlined, what stays a shared-file call, how a dropped helper gets
checked before it's trusted) are written down once in `docs/lean_conventions.md`, not re-argued per notebook.

Covers 1a-2f (sections 1 and 2 in full) plus section 3's intro so far; see the closing note at the end for
what's still open.

## 1. Load and build

1b–1d show the raw structure on a 200,000-document sample, then flatten it (`data/flat/`). 1e builds the full file (13.4M listings) from `data/raw/` (`python -m fetch.get_predictleads`) or S3 (`src/stream_full.py`). `BUILD_MODE`: **bulk** (about 35 minutes, 16 GB of RAM), **stream** (a few hours, for an 8 GB machine) or **auto**; both build the same tables, and finished steps are skipped.

In [ ]:
# 1a Libraries and the project's own data-loading code
import json
import sys
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Add analysis/ to Python's search path, so "from src..." imports work from inside notebooks/
PROJECT_ROOT = next(d for d in [Path.cwd(), *Path.cwd().parents] if (d / "src" / "pl_data.py").exists())
sys.path.insert(0, str(PROJECT_ROOT))
from src.pl_data import DATA_DIR, FLAT_DIR, describe_structure, flatten_all, load_flat, load_raw

pd.set_option("display.max_rows", 200)   # long summary tables show in full
pd.set_option("display.max_colwidth", 140)

In [ ]:
# 1b Load the raw documents ({} when every table is already flattened)
raw = load_raw()

In [ ]:
# 1c Column names and the Python type of each value in the first row
describe_structure(raw)

In [ ]:
# 1d Flatten all four tables (skipped when already done)
flatten_all(raw)

In [ ]:
# 1e Build the full-file tables (skipped when done)
from src.stream_full import build

BUILD_MODE = "auto"  # "bulk": one parallel pass | "stream": three memory-light passes | "auto": bulk when the machine can take it
build(BUILD_MODE)

## 2. Data cleaning

Full-file fixes (13.4M listings), data quality only: nothing is removed but data errors. The cleaning report (2f) lists every action.

- **Text** becomes counts and flags in the stream (`src/listing_columns.py`). Without a description a text flag is unknown, not False.
- **Dates:** impossible timelines are removed; implausible `posted` dates are blanked, and the listing stays.
- **Reposts** are kept and flagged, with features built from earlier copies only. Duplicate ids are dropped.
- **Pay** becomes one hourly rate; reversed ranges are swapped, wrong units re-read, and values outside $10k–$1M a year blanked.
- **Derived columns:** `live` (last seen within 2 days of September 30, so still open then: its lifespan is censored), `days_open`, `company_earlier` and the career-change flags.
- **Crawl artifacts** (batch first sightings, dropouts) are flagged and kept.

**Not here yet, on purpose:** this section works on all 13.4M listings, before the scope cut, so a few fields the
app needs later - `recruiter_title`, `salary_text` (pay as written), `location_text`, `url`, `record_type`,
`company_record_type` - are not loaded or computed in this notebook at all. They are fetched only for the ~900K
scope listings, in section 4 (`rows_table.scope_fields()`, joined into the scope table there) - loading them for
every listing here would cost memory for the 93% that never reach the scope.

In [ ]:
# 2a-1 Where the cleaned full-file rows live, and which columns to load
from src.rows_table import label_onehot, load_rows, row_extras

SECTION_START = time.time()  # printed in 2f-3, once section 2's cleaning is done

CLEAN_DIR = DATA_DIR / "clean"
CLEAN_DIR.mkdir(exist_ok=True)

# Every column here is listing_columns.lean_part()'s output (src/listing_columns.py: COLS + the derived text flags),
# minus id/title - row_extras() below reads those two directly to build id_fp/repost_fp instead, so the raw text
# never needs to enter this notebook's memory.
#
# Two kinds of columns are deliberately NOT loaded here, for two different reasons:
#  - Columns that exist at this full-file scale but aren't needed by this notebook: job_title, has_recruiter,
#    salary_currency, salary_low/high (pre-USD-conversion), was_translated, last_processed - added by
#    src/stream_full.py's typed_rows(), reserved for EDA_SDA_2's feature screening.
#  - Columns that do not exist yet at all, at this (13.4M-row) scale: recruiter_title, salary_text (pay as
#    written), location_text, url, record_type, company_record_type. These get fetched only for the ~900K scope
#    listings, in SECTION 4 (4d: rows_table.scope_fields(), joined with final.join(extras)) - not here, and not
#    anywhere before then. They matter for the app (e.g. url -> the listing's "apply" link), which is exactly
#    why they are kept at all, just deferred: loading them here would cost memory for the 93% of listings that
#    never reach the scope.
LOAD_COLS = ["company_id", "first_seen", "last_seen", "posted", "status", "seniority", "language", "onet_code",
             "occupation_group", "occupation", "salary_low_usd", "salary_high_usd", "salary_unit", "has_desc",
             "has_html", "desc_words", "mentions_remote", "urgency", "years_required", "entry_wording",
             "changer_wording", "degree_required", "equivalent_experience", "n_locations", "country",
             "source_type", "platform"]

# The stream writes its row progress here; it must say "finished" before load_rows() can read every part
stream = json.loads((FLAT_DIR / "stream_rows_progress.json").read_text())
if not stream.get("finished"):
    raise RuntimeError("The rows stream hasn't finished: run `python -m src.stream_full --rows` first.")

In [ ]:
# 2a-2 Load the cleaned rows (13.4M listings, light columns only)
start = time.time()
jobs = load_rows(LOAD_COLS)
print(f"loaded {len(jobs):,} rows and {jobs.shape[1]} columns in {time.time() - start:.0f}s, "
      f"{jobs.memory_usage(deep=True).sum() / 1e9:.1f} GB in memory")

In [ ]:
# 2a-3 Each row's extra fields: id and repost fingerprints, counts, and a plain column per contract type and
# category (label_onehot: the Lean alternative to a bit-packed column, at the cost of one byte per label per row)
extras, labels = row_extras()
onehot = label_onehot(labels)
jobs = pd.concat([jobs, extras.drop(columns=["contract_bits", "category_bits"]), onehot], axis=1)
del extras, onehot

In [ ]:
# 2a-4 Each company's first country, for the comparisons in section 4
from src.listing_columns import first_country

companies = load_flat("companies", ["id", "attributes.location_data"])
companies["country"] = companies.pop("attributes.location_data").map(first_country)
companies = companies.drop_duplicates("id")

In [ ]:
# 2a-5 Start the cleaning report: one row per action, filled in through section 2
report = [{"step": "2a", "action": "rows in the cleaned rows table (every English or unlabeled, US or no-country listing)",
           "rows": len(jobs)}]

In [ ]:
# 2b Standardize category labels (trim and lowercase, one column's distinct labels at a time)
for col in ["status", "seniority", "language", "salary_unit"]:
    before = jobs[col].nunique()
    jobs[col] = jobs[col].map(lambda c: str(c).strip().lower() if pd.notna(c) else c).astype("category")
    report.append({"step": "2b", "action": f"{col}: distinct labels merged by trimming and lowercasing",
                   "rows": before - jobs[col].nunique()})

# contract_types labels were already unified part by part in 2a (row_extras); just record how many merged
report.append({"step": "2b", "action": "contract_types: distinct labels merged (underscores, case, spacing)",
               "rows": labels["raw_contract_labels"] - len(labels["contract_types"])})

In [ ]:
# 2c-1 Remove listings with impossible timelines
DATA_END = pd.Timestamp("2026-09-30", tz="UTC")
END_OF_DATA = DATA_END + pd.Timedelta(days=1)

rules = {
    "missing first_seen or last_seen": jobs.first_seen.isna() | jobs.last_seen.isna(),
    "last_seen before first_seen": jobs.last_seen < jobs.first_seen,
    "dates after September 30 (the data's end date)": (jobs.first_seen > END_OF_DATA) | (jobs.last_seen > END_OF_DATA),
}
bad = pd.Series(False, index=jobs.index)
for name, mask in rules.items():
    report.append({"step": "2c", "action": f"removed, {name}", "rows": int((mask & ~bad).sum())})
    bad |= mask
jobs = jobs[~bad]
del rules

In [ ]:
# 2c-2 Blank implausible employer-posted dates (the listing itself stays)
bad_posted = (jobs.posted > jobs.first_seen + pd.Timedelta(days=1)) | (jobs.posted < pd.Timestamp("2000-01-01", tz="UTC"))
jobs.loc[bad_posted, "posted"] = pd.NaT
report.append({"step": "2c", "action": "posted blanked (after first_seen or before 2000)", "rows": int(bad_posted.sum())})

In [ ]:
# 2c-3 Drop duplicate listing ids (two different ids sharing a fingerprint is about 1-in-200,000 across 13M listings)
dup_id = jobs.id_fp.duplicated()
report.append({"step": "2c", "action": "removed, duplicate id", "rows": int(dup_id.sum())})
jobs = jobs[~dup_id].drop(columns="id_fp")

In [ ]:
# 2c-4 Flag reposts (same company, title and description as an earlier listing); kept, not dropped
jobs = jobs.sort_values("first_seen")
has_key = jobs.repost_fp.notna()
jobs["is_repost"] = has_key & jobs.duplicated("repost_fp")
report.append({"step": "2c", "action": "flagged as repost (kept, excluded from the EDA tables)",
               "rows": int(jobs.is_repost.sum())})
report.append({"step": "2c", "action": "postings without a description, can't be checked for reposts",
               "rows": int((~has_key).sum())})

In [ ]:
# 2d-1 Fix reversed salary ranges (low above high) and take the midpoint
reversed_range = jobs.salary_low_usd > jobs.salary_high_usd
jobs.loc[reversed_range, ["salary_low_usd", "salary_high_usd"]] = (
    jobs.loc[reversed_range, ["salary_high_usd", "salary_low_usd"]].to_numpy())
mid = jobs[["salary_low_usd", "salary_high_usd"]].mean(axis=1)
report.append({"step": "2d", "action": "salary present (midpoint of low and high)", "rows": int(mid.notna().sum())})
report.append({"step": "2d", "action": "salary range reversed (low above high): swapped", "rows": int(reversed_range.sum())})

In [ ]:
# 2d-2 Re-read salary units that are clearly mislabeled ("$25 a year" is really hourly, and the reverse)
PER_YEAR = {"hour": 2080, "day": 260, "week": 52, "month": 12, "year": 1}


def unit_factor(u):
    return next((f for k, f in PER_YEAR.items() if k in u), np.nan) if pd.notna(u) else np.nan


yearly_but_hourly = jobs.salary_unit.str.contains("year", na=False) & mid.between(7, 200)
hourly_but_yearly = jobs.salary_unit.str.contains("hour", na=False) & mid.between(10_000, 1_000_000)
factor = jobs.salary_unit.map(unit_factor).mask(yearly_but_hourly, PER_YEAR["hour"]).mask(hourly_but_yearly, PER_YEAR["year"])
report.append({"step": "2d", "action": "labelled yearly but clearly hourly ($7-$200): re-read as hourly",
               "rows": int(yearly_but_hourly.sum())})
report.append({"step": "2d", "action": "labelled hourly but clearly yearly ($10k-$1M): re-read as yearly",
               "rows": int(hourly_but_yearly.sum())})
report.append({"step": "2d", "action": "salary present but unit missing or not recognized",
               "rows": int((mid.notna() & factor.isna()).sum())})

In [ ]:
# 2d-3 Convert to one hourly rate and blank pay outside a plausible range ($10k-$1M a year)
HOURS_PER_YEAR = PER_YEAR["hour"]
hourly = mid * factor / HOURS_PER_YEAR
outlier = ~hourly.between(10_000 / HOURS_PER_YEAR, 1_000_000 / HOURS_PER_YEAR) & hourly.notna()
jobs["salary_hourly_usd"] = hourly.mask(outlier).astype("float32")
jobs["pay_suspect"] = reversed_range | yearly_but_hourly | hourly_but_yearly
report.append({"step": "2d", "action": "hourly pay outside $4.81-$480.77 ($10k-$1M a year), blanked", "rows": int(outlier.sum())})
report.append({"step": "2d", "action": "usable hourly pay", "rows": int(jobs.salary_hourly_usd.notna().sum())})

In [ ]:
# 2e-1 Derived timing columns: still open, days open, and the first-seen month
LIVE_DAYS = 2
jobs["live"] = (DATA_END - jobs.last_seen).dt.days <= LIVE_DAYS
jobs["days_open"] = ((jobs.last_seen - jobs.first_seen).dt.total_seconds() / 86400).astype("float32")
jobs["month"] = jobs.first_seen.dt.tz_localize(None).dt.to_period("M")
jobs["occupation_group"] = jobs.occupation_group.cat.add_categories(["(missing)"]).fillna("(missing)")

In [ ]:
# 2e-2 Career-change accessibility flags (see the decisions in section 2)
ENTRY_LEVELS = {"junior"}  # the only entry-level label (the others: mid_senior, manager, head, director,
                           # vice_president, president, c_level, partner, founder, not_set)
jobs["entry_seniority"] = jobs.seniority.isin(ENTRY_LEVELS)
jobs["low_experience_ask"] = jobs.years_required <= 2  # False where no years are stated
jobs["career_open"] = jobs.entry_seniority | jobs.low_experience_ask | jobs.entry_wording | jobs.changer_wording

In [ ]:
# 2e-3 Company history in any field, counted before the cut (a confounder for every later notebook): each
# listing's count of its company's listings first seen before it (same-day listings count together; reposts included)
jobs["company_earlier"] = (jobs.groupby("company_id", observed=True).first_seen.rank(method="min") - 1).astype("float32")

### Crawl artifacts: when the date is the crawler's, not the listing's

PredictLeads doesn't watch job postings directly - it periodically re-crawls each company's job pages and
records what it currently sees there. `first_seen` is the first time the crawler happened to notice a listing,
not necessarily when it was actually posted; `last_seen` is the last time the crawler saw it, not necessarily
when it closed. Almost always these track reality closely. Two patterns don't, and both are caused by the
crawler, not by hiring:

- **`batch_first_seen`:** PredictLeads starts crawling a company for the first time, or the company rebuilds its
  careers page and gets re-indexed - and dozens of jobs that already existed all get a `first_seen` of that one
  day. It looks like a hiring surge; it's really the crawler catching up.
- **`crawl_dropout`:** PredictLeads loses track of a company's page (a crawler bug, a site change it can't
  parse), and a batch of that company's listings all get a `last_seen` of that one day. It looks like they all
  closed together; `days_open` for those listings may be cut artificially short.

**The one-day batch rule** detects both the same way: a company with at least 5 listings, where 60% or more of
them share one exact day for the date in question (`first_seen` for batches, `last_seen` for dropouts), gets
flagged. Those two numbers (5, 60%) are a judgment call, not derived from anything - meant to catch an obvious
pile-up, not flag every small coincidence. The data's first two days are excluded from batch detection entirely:
at the very start of the crawl, *every* company's existing backlog gets dated to day one, which would make every
company look batched - that's a dataset-wide startup effect, not the per-company signal this rule is for.

Both get **flagged, kept, never dropped** - they're still real listings, just ones where a timing-based measure
(how many jobs a company posted on one day, or how long a listing stayed open) might read oddly for that
company.

In [ ]:
# 2e-4 The one-day batch rule (see the markdown above): flags listings on a day holding 60%+ of their
# company's eligible listings (5+ of them)
BATCH_MIN, BATCH_SHARE = 5, 0.6


def one_day_batch(company_codes, day, eligible):
    """True for listings on the day holding BATCH_SHARE+ of their company's eligible listings (BATCH_MIN+ of them)."""
    d = pd.DataFrame({"co": company_codes[eligible], "day": day[eligible]})
    d = d[d.co >= 0]
    per_day = d.groupby(["co", "day"]).size()
    per_co = per_day.groupby(level="co").sum()
    top = per_day.sort_values(ascending=False).groupby(level="co").head(1)  # the busiest day per company
    top = top[(per_co.reindex(top.index.get_level_values("co")).to_numpy() >= BATCH_MIN)
              & (top.to_numpy() / per_co.reindex(top.index.get_level_values("co")).to_numpy() >= BATCH_SHARE)]
    hit = pd.MultiIndex.from_arrays([company_codes, day]).isin(top.index)
    return pd.Series(hit & eligible, index=jobs.index)

In [ ]:
# 2e-5 Apply the batch and dropout rules
codes = jobs.company_id.cat.codes.to_numpy()
first_day = (jobs.first_seen.astype("int64") // 86_400_000_000_000).to_numpy()
last_day = (jobs.last_seen.astype("int64") // 86_400_000_000_000).to_numpy()
jobs["batch_first_seen"] = one_day_batch(codes, first_day, (jobs.first_seen >= pd.Timestamp("2026-04-01", tz="UTC")).to_numpy())
jobs["crawl_dropout"] = one_day_batch(codes, last_day, (~jobs.live).to_numpy())
del codes, first_day, last_day

In [ ]:
# 2e-6 How much the crawl artifacts affect the scope, in each company's own terms
BIG_COMPANIES = (jobs.company_id.value_counts() >= BATCH_MIN).sum()
artifact_stats = {
    "listings first seen in a company's one-day batch": jobs.batch_first_seen.mean(),
    "companies with such a batch (of companies with 5+ listings)":
        jobs.loc[jobs.batch_first_seen, "company_id"].nunique() / max(BIG_COMPANIES, 1),
    "listings that disappeared in a company's one-day dropout": jobs.crawl_dropout.mean(),
    "companies with such a dropout (of companies with 5+ listings)":
        jobs.loc[jobs.crawl_dropout, "company_id"].nunique() / max(BIG_COMPANIES, 1),
}

In [ ]:
# 2e-7 Save every cleaned listing to disk, then read back the analysis table (originals only, reposts excluded)
CLEANED_PATH = CLEAN_DIR / "cleaned_rows_full.parquet"
jobs.to_parquet(CLEANED_PATH)
JOBS_COLS = ["company_id", "first_seen", "occupation_group", "onet_code", "is_repost", "language", "country",
             "has_desc", "mentions_remote"]
jobs = jobs[JOBS_COLS + [c for c in jobs.columns if c.startswith("contract_")]]
jo = pd.read_parquet(CLEANED_PATH, filters=[("is_repost", "==", False)])
report.append({"step": "2e", "action": "postings in the analysis table (originals only)", "rows": len(jo)})

In [ ]:
# 2f-1 The cleaning report: every action, in order, with its share of the starting count
from IPython.display import display

cleaning_report = pd.DataFrame(report)
cleaning_report["share of start"] = (cleaning_report.rows / report[0]["rows"]).round(4)
print("Cleaning report (every action, in order)")
display(cleaning_report)

In [ ]:
# 2f-2 The analysis table at a glance (originals only: the rows for the every-field comparison)
print("Analysis table at a glance (originals: the rows for the every-field comparison)")
display(pd.Series({
    "postings": f"{len(jo):,}",
    "companies": f"{jo.company_id.nunique():,}",
    "occupation groups": jo.occupation_group.nunique(),
    "still open on September 30": f"{jo.live.mean():.1%}",
    "first_seen from": jo.first_seen.min().date(),
    "first_seen to": jo.first_seen.max().date(),
}).to_frame("value"))

In [ ]:
# 2f-3 The crawl artifacts, and how long section 2 took
print("Crawl artifacts (flagged, kept): every cleaned listing")
display(pd.Series(artifact_stats).to_frame("share"))
print(f"section 2 done in {(time.time() - SECTION_START) / 60:.1f} min")

## 3. Q1: How do computer and mathematical roles compare with every other field?

Every occupation group before the scope cut, on original English listings; weekly rates use full weeks only.
**Context, not cause:** fields differ in company size, source and seniority.

### Why this needs Kaplan-Meier

**22.3% of all original listings were still open on September 30** - not a tail case, roughly a quarter of the
data - and it isn't spread evenly: a listing posted in late September has barely had time to close, so it's far
more likely to still be open than one posted in April. Any "typical time to close" or "closed within 7 days"
computed only from listings that have already closed would read short, worst for the most recently posted
listings in every field. `median days open` and `closed within N days` below use the Kaplan-Meier curve
(`src/kaplan_meier.py`, `docs/kaplan_meier.md`) for exactly this reason, computed within each occupation group
from its closed *and* still-open listings together - not just the ones that happen to still be open.

This isn't a one-off, either: `TimeSeries` runs Kaplan-Meier by month for the same reason (each month's cohort
has a different amount of follow-up); `EDA_SDA_4` fits a Cox model; `ML_Rank`'s priority queue trains an actual
survival forest (C-index 0.725) for its "chance still open" factor. None of those are possible without a method
that uses the closed and still-open listings together.

---

### Where this draft stops, and why

1a-2f are now rewritten in full (section 1, load and build; section 2, cleaning): library setup, loading the
cleaned rows, plain one-hot contract/category columns, label standardizing, date cleaning, duplicate/repost
flags, salary conversion, derived timing and career-change flags, company history, crawl artifacts, and the
cleaning report. `relabel`, `cat_contains`, `cat_map`, `with_missing`, `show`/`show_shaded`/`compact`/`pct` and
`section_done()` are not used anywhere in this notebook; see the cells above for what replaced each one.

Section 3's own intro (above) is drafted, including why it needs Kaplan-Meier rather than a plain average - that
question is resolved (`src/kaplan_meier.py`, `docs/kaplan_meier.md`, checked against `src/eda_helpers.py`'s
version in `tests/test_kaplan_meier.py`). Not yet drafted: 3a-3c's actual code (the shared constants, the
every-field comparison table, the chart), which still needs the chart-style constants (`BLUE`, `ORANGE`,
`SURFACE`, ...) from `src/eda_helpers.py` - trivial to inline, just not done yet.

**Not yet run on the real file.** The "What the data shows" findings text from `EDA_SDA_1`'s section 2 isn't
copied here: the numbers it quotes (1.4M reposts, 349K blanked dates, etc.) describe `EDA_SDA_1`'s run, not this
draft's, and shouldn't be presented as this notebook's own findings until it's actually run and checked. The
22.3% figure quoted above is also `EDA_SDA_1`'s, cited as the reason the method is used, not yet confirmed for
this draft's own run.